# Generalized Disjunctive Programming

[Open this notebook in Colab](https://colab.research.google.com/github/SECQUOIA/pyomo-summer-ws/blob/main/notebooks/06-gdp/GDP.ipynb).

This notebook is part of the [SECQUOIA Research Group](https://engineering.purdue.edu/SECQUOIA) Pyomo Tutorial. It adapts the GDP strip-packing exercise from the public Pyomo workshop tutorials. The transformed examples use GLPK; see [Setup and Solvers](../../setup.md) for solver notes.


## Learning objectives

By the end of this chapter, you should be able to:

- express logical alternatives with `Disjunction` objects;
- explain why GDP keeps modeling intent clearer than hand-written big-M constraints;
- transform a GDP model to a mixed-integer formulation with big-M or hull reformulations;
- solve a transformed GDP model with a MIP solver;
- inspect which disjuncts are active in the solved model.


In [1]:
import pyomo.environ as pyo
from pyomo.gdp import Disjunction


def solve_with_glpk(model):
    solver = pyo.SolverFactory("glpk")
    if not solver.available(False):
        print("GLPK is not available; the model was built but not solved.")
        return None
    results = solver.solve(model)
    print(f"termination: {results.solver.termination_condition}")
    return results


## 1. Logical alternatives in algebraic models

GDP is useful when a model has explicit alternatives: one unit is active or bypassed, one rectangle is left of another or below it, or one operating mode is selected. Pyomo lets you state those alternatives directly as disjunctions, then transform the model for a solver.

The workshop exercise packs rectangles into a strip of fixed width while minimizing strip length. For each pair of rectangles, at least one no-overlap relation must hold.

### Exercise 1.1. Build the strip-packing GDP model

Start from [`stripPacking.py`](../exercises/GDP/exercises-1/stripPacking.py) and insert the no-overlap disjunctions. A complete solution is available in [`stripPacking_soln.py`](../exercises/GDP/exercises-1/stripPacking_soln.py).


In [2]:
def build_strip_packing_model():
    model = pyo.ConcreteModel()
    model.RECTANGLES = pyo.Set(ordered=True, initialize=[0, 1, 2, 3])
    model.Width = pyo.Param(model.RECTANGLES, initialize={0: 6, 1: 3, 2: 4, 3: 2})
    model.Length = pyo.Param(model.RECTANGLES, initialize={0: 6, 1: 8, 2: 5, 3: 3})
    model.StripWidth = pyo.Param(initialize=10)
    model.LengthUB = pyo.Param(initialize=sum(model.Length[i] for i in model.RECTANGLES))
    model.x = pyo.Var(model.RECTANGLES, bounds=(0, model.LengthUB))

    def y_bounds(model, i):
        return (0, model.StripWidth - model.Width[i])

    model.y = pyo.Var(model.RECTANGLES, bounds=y_bounds)
    model.MaxLength = pyo.Var(within=pyo.NonNegativeReals, bounds=(0, model.LengthUB))

    def rec_pairs_filter(model, i, j):
        return i < j

    model.OVERLAP_PAIRS = pyo.Set(
        initialize=model.RECTANGLES * model.RECTANGLES,
        dimen=2,
        filter=rec_pairs_filter,
    )

    @model.Constraint(model.RECTANGLES)
    def strip_ends_after_last_rec(model, i):
        return model.MaxLength >= model.x[i] + model.Length[i]

    model.total_length = pyo.Objective(expr=model.MaxLength)

    @model.Disjunction(model.OVERLAP_PAIRS)
    def no_overlap(model, i, j):
        return [
            model.x[i] + model.Length[i] <= model.x[j],
            model.x[j] + model.Length[j] <= model.x[i],
            model.y[i] + model.Width[i] <= model.y[j],
            model.y[j] + model.Width[j] <= model.y[i],
        ]

    return model


gdp_model = build_strip_packing_model()
print(f"rectangle pairs requiring a disjunction: {len(gdp_model.OVERLAP_PAIRS)}")
print(f"disjuncts for pair (0, 1): {len(gdp_model.no_overlap[0, 1].disjuncts)}")


rectangle pairs requiring a disjunction: 6
disjuncts for pair (0, 1): 4


## 2. Transforming a GDP model

A GDP model is not sent directly to GLPK. Pyomo rewrites the disjunctions into a mixed-integer formulation. The big-M reformulation is often compact; the hull reformulation is often tighter but can create more variables and constraints.

### Exercise 2.1. Apply the big-M transformation

Because all rectangle coordinates have finite bounds, Pyomo can estimate valid M values for this small model.


In [3]:
bigm_model = build_strip_packing_model()
pyo.TransformationFactory("gdp.bigm").apply_to(bigm_model)
result = solve_with_glpk(bigm_model)

if result is not None:
    print(f"minimum strip length: {pyo.value(bigm_model.total_length):.1f}")
    for i in bigm_model.RECTANGLES:
        print(
            f"rectangle {i}: x={pyo.value(bigm_model.x[i]):4.1f}, "
            f"y={pyo.value(bigm_model.y[i]):4.1f}, "
            f"length={pyo.value(bigm_model.Length[i]):.0f}, width={pyo.value(bigm_model.Width[i]):.0f}"
        )


termination: optimal
minimum strip length: 11.0
rectangle 0: x= 0.0, y= 4.0, length=6, width=6
rectangle 1: x= 0.0, y= 0.0, length=8, width=3
rectangle 2: x= 6.0, y= 3.0, length=5, width=4
rectangle 3: x= 8.0, y= 0.0, length=3, width=2


### Exercise 2.2. Inspect selected disjuncts

The original disjunction objects remain attached to the transformed model, so you can still inspect which logical branch was selected after the solve.


In [4]:
branch_labels = ["i left of j", "j left of i", "i below j", "j below i"]

for pair in bigm_model.OVERLAP_PAIRS:
    choices = [bool(pyo.value(disjunct.indicator_var)) for disjunct in bigm_model.no_overlap[pair].disjuncts]
    selected = branch_labels[choices.index(True)] if any(choices) else "no active branch"
    print(f"pair {pair}: {selected}")


pair (0, 1): j below i
pair (0, 2): i left of j
pair (0, 3): j below i
pair (1, 2): i below j
pair (1, 3): i left of j
pair (2, 3): j below i


### Exercise 2.3. Compare big-M and hull transformations

Solve the same GDP model with the hull reformulation. Compare the objective value and the number of generated variables and constraints.


In [5]:
comparison = []
for transformation in ["gdp.bigm", "gdp.hull"]:
    model = build_strip_packing_model()
    pyo.TransformationFactory(transformation).apply_to(model)
    result = solve_with_glpk(model)
    variables = len(list(model.component_data_objects(pyo.Var)))
    constraints = len(list(model.component_data_objects(pyo.Constraint, active=True)))
    objective = pyo.value(model.total_length) if result is not None else float("nan")
    comparison.append((transformation, variables, constraints, objective))

for transformation, variables, constraints, objective in comparison:
    print(f"{transformation:8s}: vars={variables:3d}, constraints={constraints:3d}, objective={objective:.1f}")


termination: optimal


termination: optimal
gdp.bigm: vars= 33, constraints= 34, objective=11.0
gdp.hull: vars=105, constraints=130, objective=11.0


## Exercises to continue

1. Change one rectangle's width or length and resolve both transformations.
2. Add a fifth rectangle and observe how the number of rectangle-pair disjunctions grows.
3. Add rotation decisions, then decide whether the rotation alternatives should be modeled with additional binaries or GDP disjunctions.

## References

GDP models and transformations are part of Pyomo's modeling extensions [@bynum2021pyomo].
